# Control de la 3.2.3: reglas fija y uniforme (liquidez media, cuaderno B)

**Quién lo corre:** Paolo (PS), junto con el cuaderno A. **No** hay que repetir nada de las 90 corridas: esto se suma a ellas.

**Qué hace:** corre el sistema con el Maestro reemplazado por dos reglas, con los **mismos Ejecutores, órdenes y semillas** (4, 5 y 6) de las 90 corridas, en MO-06, MO-07, MO-08, MO-09, MO-10:
- **Regla fija:** α = 0,5 y ventana de 15 min siempre (lo que el Maestro elige en el 98 % de los casos).
- **Regla uniforme:** partes iguales cada media hora y 15 min, es decir, el calendario de TWAP ejecutado por los Ejecutores PPO.

Con eso se separa cuánto de la diferencia con TWAP viene del Maestro y cuánto de los Ejecutores.

**Cómo usarlo:**
1. Abre este cuaderno y el A **al mismo tiempo** y en cada uno usa «Entorno de ejecución → Ejecutar todas».
2. Cada uno dura **≈ 2 horas**. Si Colab se corta, vuelve a «Ejecutar todas»: retoma donde quedó.
3. La celda 13 debe decir **LISTO**.
4. El que termine último hace el análisis (celda 14) y el zip `control_reglas_media.zip` (celda 15). Ese zip se lo pasas a Claude.

**Necesita:** los checkpoints del sistema final (`slippage_colab/resultados/sistema_final/maestro/`) y la carpeta `slippage_colab/resultados/corridas_90/media/` de las 90 corridas (ya están en tu Drive).

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama feature/ps-3-2-3-control-reglas, o main si ya se mergeó; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && (git checkout -q -f origin/feature/ps-3-2-3-control-reglas || git checkout -q -f origin/main)
!cd /content/slippage && git log --oneline -1 && grep -c "def accion_regla" scripts/colab/evaluar_sistema_ps.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Parámetros

In [ ]:
# CELDA 10: PARÁMETROS (los mismos de las 90 corridas: no cambiarlos)
ESCENARIO = 'media'
PARTE = 'B'                # A = MO-01 a MO-05, B = MO-06 a MO-10
ORDENES = 'MO-06 MO-07 MO-08 MO-09 MO-10'
SEMILLA_INICIAL = 4         # las mismas semillas 4, 5 y 6 de las 90 corridas: así se puede parear
SEMILLAS = 3
OUT = f'{RESULTS_DIR}/control_reglas/{ESCENARIO}/{PARTE}'
print('Control | escenario', ESCENARIO, '| parte', PARTE, '| órdenes', ORDENES, '| carpeta', OUT)


## Corridas

In [ ]:
# CELDA 11: regla fija (α = 0,5 y 15 min siempre). 15 corridas, ≈ 1 h. Si se corta, vuelve a ejecutar y retoma donde quedó
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/evaluar_sistema_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --ordenes {ORDENES} --seeds {SEMILLAS} --semilla-inicial {SEMILLA_INICIAL} --escenario {ESCENARIO} --escenarios-json data/scenarios/escenarios_liquidez_v3.json --impacto --out-dir {OUT} --max-minutes 330 --completar-al-cierre --modo muestreo --ckpt-maestro {RESULTS_DIR}/sistema_final/maestro/checkpoint_last.pt --ckpt-ejecutor apertura={RESULTS_DIR}/sistema_final/maestro/ejecutor_apertura.pt --ckpt-ejecutor media_jornada={RESULTS_DIR}/sistema_final/maestro/ejecutor_media_jornada.pt --ckpt-ejecutor cierre={RESULTS_DIR}/sistema_final/maestro/ejecutor_cierre.pt --regla fija 2>&1 | grep -v UserWarning


In [ ]:
# CELDA 12: regla uniforme (el calendario de TWAP, con los Ejecutores PPO). 15 corridas, ≈ 1 h. Si se corta, vuelve a ejecutar y retoma donde quedó
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/evaluar_sistema_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --ordenes {ORDENES} --seeds {SEMILLAS} --semilla-inicial {SEMILLA_INICIAL} --escenario {ESCENARIO} --escenarios-json data/scenarios/escenarios_liquidez_v3.json --impacto --out-dir {OUT} --max-minutes 330 --completar-al-cierre --modo muestreo --ckpt-maestro {RESULTS_DIR}/sistema_final/maestro/checkpoint_last.pt --ckpt-ejecutor apertura={RESULTS_DIR}/sistema_final/maestro/ejecutor_apertura.pt --ckpt-ejecutor media_jornada={RESULTS_DIR}/sistema_final/maestro/ejecutor_media_jornada.pt --ckpt-ejecutor cierre={RESULTS_DIR}/sistema_final/maestro/ejecutor_cierre.pt --regla uniforme 2>&1 | grep -v UserWarning


## Revisión

In [ ]:
# CELDA 13: revisión (deben salir 15 de cada regla, con cumplimiento >= 0,95)
import json, glob
esperado = {'Regla_fija': 15, 'Regla_uniforme': 15}
conteo, malas = {}, []
for f in glob.glob(f'{OUT}/*.json'):
    for c in json.load(open(f, encoding='utf-8')).get('corridas', {}).values():
        if 'error' in c:
            malas.append((c['orden'], c['estrategia'], c['seed'], c['error'][:80])); continue
        conteo[c['estrategia']] = conteo.get(c['estrategia'], 0) + 1
        if c.get('execution_rate', 1.0) < 0.95:
            malas.append((c['orden'], c['estrategia'], c['seed'], f"cumplimiento {c['execution_rate']:.3f}"))
print('Corridas:', conteo, '(se esperan', esperado, ')')
print('Problemas:', malas if malas else 'ninguno')
print('LISTO' if conteo == esperado and not malas else 'FALTA ALGO: vuelve a ejecutar las celdas 11 y 12 (retoman donde quedaron)')


## Análisis (cuando A y B terminaron)

In [ ]:
# CELDA 14: análisis (solo cuando A y B terminaron). Usa también las 90 corridas de la media que ya están en tu Drive
import os
base = f'{RESULTS_DIR}/control_reglas/{ESCENARIO}'
media90 = f'{RESULTS_DIR}/corridas_90/{ESCENARIO}'
listos = [p for p in ('A', 'B') if len(glob.glob(f'{base}/{p}/*.json')) == 2]
print('Partes con sus 2 archivos:', listos, '| media de las 90 en Drive:', os.path.isdir(media90))
if len(listos) == 2 and os.path.isdir(media90):
    get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/control_reglas_3_2_3.py --entradas {media90}/A {media90}/B {base}/A {base}/B --out-dir {base}/analisis")
else:
    print('Falta el otro cuaderno (o la carpeta corridas_90/media): el análisis y el zip salen del que termine último.')


## Zip para mandar

In [ ]:
# CELDA 15: zip para mandar (sale completo del cuaderno que termina último)
import zipfile
archivos = [os.path.join(c, a) for c, _, fs in os.walk(base) for a in fs if a.endswith(('.json', '.md', '.png'))]
zpath = f'{RESULTS_DIR}/control_reglas_{ESCENARIO}.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print(len(archivos), 'archivos -> zip listo:', zpath, '(descárgalo desde slippage_colab/resultados en Drive)')
